In [1]:
import pandas as pd
import numpy  as np
from datetime import datetime

#################################################
# cln_imdb__xref_titles_principals_filtered,  PK (nconst, tconst)
#################################################

P = pd.read_csv('https://datasets.imdbws.com/title.principals.tsv.gz'
                 , compression='gzip'
                 , delimiter='\t',encoding='utf-8'
                 , header=0
                # , sep= " "
                 , quotechar='"'
                 #, error_bad_lines=False
                 , dtype={ "tconst": "str" 
                         , "ordering": "str"
                         , "nconst": "str" 
                         , "category": "str" 
  # save space           , "job": "str"
  # save space           , "characters": "str"         
                         }
                )

print ('read/clean p1, takes ages and requires a lot of memory')
# exclude rows not necessarily needed
P["cat_cln"] = np.where (P['category'].str.contains('production_designer') 
                       | P['category'].str.contains('archive_sound') 
                       | P['category'].str.contains('casting_director')
                       | P['category'].str.contains('director')  
                         , 'ex'
                         ,P['category'])

 

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'P: ',  P.shape)

P = P[P['cat_cln']!= 'ex'] 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'P_cln: ',  P.shape)


# group values actress, actor to actor
P["cat_cln"] = np.where (P['category'].str.contains('actor') | P['category'].str.contains('actress')
                         ,'actor'
                         ,P['category'])

# group values self, archive_footage to self
P["cat_cln"] = np.where (P['category'].str.contains('self') | P['category'].str.contains('archive_footage')
                         ,'self',P['cat_cln'])

#P = P[P['category']!= "archive_sound"] 
#P = P[P['category']!= "casting_director"]



print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'P: ',  P.shape)
P.head(5)

read/clean p1, takes ages and requires a lot of memory
2026-06-30 16:25:05 P:  (100235829, 7)
2026-06-30 16:25:57 P_cln:  (89363209, 7)
2026-06-30 16:27:40 P:  (89363209, 7)


,tconst,ordering,nconst,category,job,characters,cat_cln
0,tt0000001,1,nm1588970,self,\N,"[""Self""]",self
2,tt0000001,3,nm0005690,producer,producer,\N,producer
3,tt0000001,4,nm0374658,cinematographer,director of photography,\N,cinematographer
5,tt0000002,2,nm1335271,composer,\N,\N,composer
7,tt0000003,2,nm0721526,writer,\N,\N,writer


In [2]:
import pandas as pd
import numpy  as np
#################################################
# Titles crew directors writers, missing from principal file
#################################################

Tcrew = pd.read_csv('https://datasets.imdbws.com/title.crew.tsv.gz'
                 , compression='gzip'
                 , delimiter='\t'
                 , encoding='utf-8'
                 , header=0
                # , sep= " "
                 , quotechar='"'  
                 , dtype={ "tconst": "str" ,
                           "directors": "str" ,
                           "writers": "str" ,       
                         }   
                ) 

Tcrew.rename(columns = {  "directors": "nconst"}
                      , inplace = True)
Tcrew.head(5)
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

2026-06-30 16:28:17


In [3]:
#Tcrew[Tcrew['tconst'] == "tt0000335"]['nconst'].str.split(',').str[0]
#Tcrew[Tcrew['tconst'] == "tt0000335"]['nconst'].str.split(',').str[1]
#Tcrew[Tcrew['tconst'] == "tt1745960"]['nconst'].str.split(',').str[0]
#Tcrew[Tcrew['tconst'] == "tt1745960"]['nconst'].str.split(',').str[1] 
Tcrewi= pd.DataFrame(columns = ['tconst', 'ordering', 'nconst', 'category', 'job', 'characters', 'cat_cln'])
Tcrew9= pd.DataFrame(columns = ['tconst', 'ordering', 'nconst', 'category', 'job', 'characters', 'cat_cln'])
for i in range(0, 4):

    print(i)
    Tcrewi['tconst']   = Tcrew['tconst'] 
    Tcrewi['ordering'] =i
    Tcrewi['nconst']   = Tcrew['nconst'].str.split(',').str[i]
    Tcrewi['category'] ='director'
    Tcrewi['cat_cln']  ='director'
    Tcrewi             =Tcrewi.dropna(subset=['nconst'])
    print (Tcrewi.shape)
    Tcrew9 =pd.concat([Tcrew9, Tcrewi], sort=False)
    print (Tcrew9.shape)
    
#Tcrew9=Tcrew9.dropna(subset=['nconst'])
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

0
(12608722, 7)
(12608722, 7)
1
(1398124, 7)
(14006846, 7)
2
(462267, 7)
(14469113, 7)
3
(211266, 7)
(14680379, 7)
2026-06-30 16:30:17


In [4]:
# add crew to principals
#category == 'director'
#var_list = ('tconst' , 'ordering', 'nconst', 'category', 'job' , 'characters' , 'cat_cln') 


#P1 = P[P['category']!= "director"] 
#P1=pd.concat([P1, Tcrew9], sort=False)
#P1.head(3)

# P = P[P['category']!= "director"]   already excluded in prev steüp
P = pd.concat([P, Tcrew9], sort=False)
P.head(3)
print(P.shape)
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

(104043588, 7)
2026-06-30 16:31:21


In [5]:
Tcrew[Tcrew['tconst'] == "tt1024648"].head(15)

,tconst,nconst,writers
1103477,tt1024648,nm0000255,"nm0006516,nm5410196,nm2646452"


In [6]:
print (P.shape)
P = P.drop_duplicates(subset= ['tconst','nconst'], keep="last")
print (P.shape)
P = P.drop(columns=['job', 'characters'])
print (P.shape)
#P1[P1['tconst'] == "tt0000335"].head(15)
P[P['tconst'] == "tt1024648"].head(15)
#Tcrew9[Tcrew9['tconst'] == ""].head(15)
#P1.head(5)

#Tdir[Tdir['tconst'] == "tt1745960"].head(15)
#Tcrew9.head(7)

(104043588, 7)
(94787298, 7)
(94787298, 5)


,tconst,ordering,nconst,category,cat_cln
12803220,tt1024648,2,nm0186505,actor,actor
12803221,tt1024648,3,nm0000422,actor,actor
12803222,tt1024648,4,nm0000273,actor,actor
12803223,tt1024648,5,nm0001255,actor,actor
12803224,tt1024648,6,nm0004883,actor,actor
12803225,tt1024648,7,nm0245112,actress,actor
12803226,tt1024648,8,nm1058940,actor,actor
12803227,tt1024648,9,nm0168262,actor,actor
12803228,tt1024648,10,nm1706832,actor,actor
12803230,tt1024648,12,nm0006516,writer,writer


In [7]:
# P --> cln_imdb__xref_titles_principals_filtered

import os

data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
output_file = os.path.join(data_path, "dwh_xref_titles_principals__filtered.csv")

P.to_csv(output_file, encoding="utf-8")